# Explicit dyes: labelling, rotamer libraries and FRET

`IMP.bff` models a fluorescent label in two ways. The **accessible volume** (AV,
see *Accessible volumes*) treats the dye as a point on a flexible linker and
enumerates where it can go. The **explicit dye** route (the `cgdye` layer)
places an atomistic dye + linker on a residue, either from a **rotamer library**
(FRETpredict's libraries ship as module data) or by **sampling the linker
degrees of freedom**, and scores each conformer against the protein. This page
walks the explicit route with the flat `IMP.bff.*` API; everything shown here is
reachable as `IMP.bff.<Name>` (the sub-packages are just where the code lives).

Physics pins behind this page (PRD-107): rotamer FRET reproduces FRETpredict on
its Hsp90 and pp11 references (`test/cgdye/rotamer/test_fretpredict_pins.py`),
and the invariants of κ², R0, the FRET regimes and the kinetic master equation
are asserted in `test/cgdye/test_physics_invariants.py`.


## 1. Label a structure with an explicit dye

`attach_dyes` moves a dye hierarchy (read from MOL2 or PDB) into the backbone
frame of a residue (origin CA, x along CA→N, y in the N–CA–C plane) and, on
request, strips the residue's side chain first. Note the keep-set: cgdye keeps
`N CA C O OXT` and **strips CB**, because the explicit linker is built off CA
and replaces the whole side chain; the AV convention keeps CB. Both go through
one strip engine (`IMP.bff.strip_hierarchy`, PRD-106).


In [ ]:
import IMP, IMP.atom, IMP.core
import IMP.bff
from IMP.bff.cgdye.utils import get_structure_dir   # bundled inputs (data/cgdye)

m = IMP.Model()
protein = IMP.atom.read_pdb(str(get_structure_dir("1DG3.pdb")), m, IMP.atom.NonWaterPDBSelector())
dye = IMP.atom.read_mol2(str(get_structure_dir("alexa488_r48.mol2")), m)

site = IMP.bff.resolve_dye_site(protein, "A", 481)          # {'CA':..., 'N':..., 'C':...}
attached = IMP.bff.attach_dyes(protein, [(dye, "A", 481)], strip_site_sidechain=True)
print("dye atoms:", len(IMP.atom.get_by_type(dye, IMP.atom.ATOM_TYPE)),
      "site keep-set:", IMP.bff.SITE_KEEP_ATOM_NAMES)


## 2. Rotamer libraries

The FRETpredict rotamer libraries are IMP.bff module data
(`data/rotamer_library`: `<stem>.pdb` + `<stem>_cutoff<N>.dcd` + weights). A
library name selects the dye, the linker and the clustering cutoff; the cutoff
in the name is honoured (`cutoff10` = 711 rotamers for Alexa488 C1R, `cutoff30`
= 33 — the default).


In [ ]:
lib = IMP.bff.load_rotamer_library("AlexaFluor 488 C1R cutoff30")
print(lib["coords"].shape, "rotamers x atoms x 3;", "weights sum", lib["weights"].sum())
print("chromophore centre selector:", lib["metadata"]["r"], " dipole:", lib["metadata"]["mu"])
print(IMP.bff.resolve_rotamer_library_path("AlexaFluor 488 C1R cutoff10").name)


## 3. Förster radius and κ²

R0 comes from the bundled donor emission / acceptor excitation spectra
(`IMP.bff.forster_radius_from_spectra`, nm), κ² from transition-dipole
vectors (`IMP.bff.kappa2_from_dipoles`); both live in `IMP.bff.fret` because
they are properties of the label pair, not of a coordinate model.


In [ ]:
import numpy as np
r0_iso = IMP.bff.forster_radius_from_spectra("AlexaFluor 488", "AlexaFluor 594", IMP.bff.kappa2_isotropic())
print("R0(A488/A594, kappa2=2/3) =", round(r0_iso, 3), "nm")
mu_d = np.array([[0.0, 0.0, 1.0]]); mu_a = np.array([[0.0, 0.0, 1.0]]); r = np.array([[[0.0, 0.0, 50.0]]])
print("collinear kappa2 =", IMP.bff.kappa2_from_dipoles(mu_d, mu_a, r)[0, 0])


## 4. Rotamer FRET (FRETpredict-compatible)

`RotamerFRET` places two libraries on two sites of a structure (PDB, multi-MODEL
PDB, or RMF trajectory), screens every rotamer against the protein
(Lennard-Jones, optional Debye–Hückel), and reports per frame the static,
dynamic-1 and dynamic-2 efficiencies, ⟨κ²⟩ and the partition functions.


In [ ]:
from IMP.bff.cgdye.utils import get_structure_dir
fret = IMP.bff.RotamerFRET(
    str(get_structure_dir("148L.pdb")), [22, 137], chains=["E", "E"],   # T4 lysozyme, chain E
    donor="AlexaFluor 488", acceptor="AlexaFluor 594",
    libname_1="AlexaFluor 488 C1R cutoff30", libname_2="AlexaFluor 594 C1R cutoff30",
    temperature=298, electrostatic=True, output_prefix="t4l_rotamer")
fret.trajectory_analysis()
print("E_static", fret.estatic_values, "E_dyn1", fret.edynamic1_values, "<kappa2>", fret.k2_values, "Z", fret.z_values)


## 5. FRET regimes and the kinetic master equation

Given a donor×acceptor grid of distances and κ² with weights,
`fret_efficiency_regimes` returns the static, dynamic and dynamic+ averages;
`fret_efficiency_exact_kinetic` solves the master equation for a transition
matrix (slow exchange → static average, fast exchange → dynamic average).


In [ ]:
d = np.array([[45.0, 60.0], [52.0, 48.0]]); k2 = np.array([[0.5, 1.2], [0.9, 2.0]])
wd = np.array([0.7, 0.3]); wa = np.array([0.4, 0.6])
print(IMP.bff.fret_efficiency_regimes(d, k2, wd, wa, R0=52.0))
P = IMP.bff.rotamer_transition_matrix([[8, 2], [3, 7]])
rates = ((52.0 / d[:, 0]) ** 6 * 1.5 * k2[:, 0] / 4.0)      # donor states vs one acceptor, 1/ns
print("E (kinetic, donor exchange):", IMP.bff.fret_efficiency_exact_kinetic(P, rates, tau0=4.0, dt=0.1))


## 6. Sampling the linker yourself

`LinkerSampler` / `generate_linker_rotamers` Metropolis-sample a dye's linker
torsions and bond angles (bonded 1-2/1-3/1-4 pairs excluded from the LJ score),
cluster the frames and Boltzmann-weight the clusters; `run_torsion_rrt` and
`run_rigid_body_rrt` grow collision-free trees; the command line
(`dye --help`) drives the same code. Langevin/Brownian dynamics of the dye
arrives with PRD-108.


In [ ]:
lib_gen = IMP.bff.generate_linker_rotamers(str(get_structure_dir("alexa488_r48.mol2")),
                                          n_steps=200, write_every=10, cluster_threshold=1.0, seed=42)
print(len(lib_gen["weight"]), "clusters; transitions", np.array(lib_gen["transitions"]).sum())
